# 04a - Preregistered Validation-Adaptive Ensemble Research

This notebook performs **no training**. It evaluates exactly one preregistered policy on immutable historical cross-predictions that end before both failed Future-OOS windows. Model selection and threshold calibration use disjoint chronological validation subsets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_BASE = '/content/drive/MyDrive/yeniBot'
CHECKPT_DIR = f'{DRIVE_BASE}/checkpoints'
REPORT_DIR = f'{DRIVE_BASE}/reports'
os.makedirs(CHECKPT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

In [ ]:
import os, subprocess, sys
REPO_URL = 'https://github.com/umutergul74/yeniBot.git'
REPO_DIR = '/content/yenibot_repo'
REPO_BRANCH = os.environ.get('YENIBOT_REPO_BRANCH', 'codex/phase1-research-v2')
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-B', REPO_BRANCH, f'origin/{REPO_BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, REPO_DIR], check=True)
repo_commit = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
repo_branch = subprocess.check_output(['git', '-C', REPO_DIR, 'branch', '--show-current'], text=True).strip()
assert repo_branch == REPO_BRANCH, f'Expected {REPO_BRANCH}, found {repo_branch}'
sys.path.insert(0, REPO_DIR)
print('Repository branch:', repo_branch)
print('Preregistration commit:', repo_commit)

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
import json, shutil, yaml
from pathlib import Path
from yenibot.experiment import validate_training_research_contract, run_cached_adaptive_ensemble_research
with open(f'{REPO_DIR}/config.yaml', encoding='utf-8') as handle:
    cfg = yaml.safe_load(handle)
contract = validate_training_research_contract(cfg)
research = cfg['experiments']['next_research_cycle']['adaptive_ensemble']
assert contract['adaptive_ensemble_enabled'] is True
assert research['preregistered'] is True
source_run_id = str(research['source_run_id'])
hypothesis_id = str(research['hypothesis_id'])
source_dir = Path(CHECKPT_DIR) / 'experiments' / source_run_id / 'recency_research'
output_dir = Path(CHECKPT_DIR) / 'policy_research' / hypothesis_id
print('Hypothesis:', hypothesis_id)
print('Immutable source cache:', source_dir)
print('Separate output directory:', output_dir)
result = run_cached_adaptive_ensemble_research(
    source_research_dir=source_dir,
    output_dir=output_dir,
    config=cfg,
    code_commit=repo_commit,
)
print('Research status:', result['status'])
print('Fit operations performed:', result['manifest']['fit_operations_performed'])
print('Failed Future-OOS used for selection:', result['manifest']['failed_future_oos_used_for_selection'])
display(result['summary'])
display(result['paired_comparison'])
print(json.dumps(result['decision'], indent=2))
handoff = {
    'workflow': 'phase1_preregistered_adaptive_policy_research',
    'repo_branch': repo_branch,
    'repo_commit': repo_commit,
    'hypothesis_id': hypothesis_id,
    'source_run_id': source_run_id,
    'output_dir': str(output_dir),
    'decision_status': result['decision'].get('status'),
    'candidate_ready_for_preregistration': result['decision'].get('candidate_ready_for_preregistration', False),
    'next_action': ('send_policy_research_bundle_for_review' if result['decision'].get('candidate_ready_for_preregistration', False) else 'archive_failed_hypothesis_do_not_run_notebook_05'),
}
handoff_path = Path(CHECKPT_DIR) / 'notebook04a_policy_research.json'
handoff_path.write_text(json.dumps(handoff, indent=2), encoding='utf-8')
archive_base = Path(REPORT_DIR) / f'phase1_policy_research_{hypothesis_id}'
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=output_dir))
latest_path = Path(REPORT_DIR) / 'phase1_latest_policy_research_bundle.zip'
shutil.copy2(archive_path, latest_path)
print('Bundle:', latest_path)
print('IMPORTANT: Do not run Notebook 05 yet. Send this bundle for review.')